# Adding Sinks to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Sink components**. Sink represent components that **absorb a commodity and remove it from the energy system**. We focus on the most essential parameters required to define and understand a Sink component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of sinks include:

- electricity or heat demand
- CO2 released into the environment


Sink therefore represent **exit points of commodities from the modeled system**.

The Sink component in FINE is closely related to the Source component and internally inherits from the same class. Therefore, the parameters described here are also relevant when modeling sinks.


## Load ESM

We first load the ESM from the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb).

In [1]:
import fine as fn
import fine.IOManagement.xarrayIO as xrIO
import pandas as pd
import numpy as np
from pathlib import Path
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source.nc"

esM = xrIO.readNetCDFtoEnergySystemModel(nc_file)

## Add Sinks

### Electricity demand

We can now add an electricity demand as a first sink.

To do so we first generate a normalized daily load shape (`dailyProfile`) which defines typical relative demand levels for each hour of the day. This profile is repeated for 365 days, and small random variations (up to $+0.1$) are added to each hourly value to introduce variability. The resulting values are then scaled to approximate demand levels for two regions(`regionN` and `regionS`) using different multipliers (25 and 40, respectively). The final DataFrame is rounded to two decimal places and indexed by hourly timesteps.

In [2]:
dailyProfile = [
    0.6, # 12am - 1am
    0.6, # 1am - 2am
    0.6, # ...
    0.6,
    0.6,
    0.7,
    0.9,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0.9,
    0.8 # 11pm - 12 am
    ]

electricityDemand = pd.DataFrame(
    [
        [(u + 0.1 * np.random.rand()) * 25, (u + 0.1 * np.random.rand()) * 40]
        for day in range(365)
        for u in dailyProfile
    ],
    index=range(8760), # Timesteps of the esM
    columns=["regionN", "regionS"],
).round(2)

esM.add(
    fn.Sink(
        esM = esM,
        name = "Electricity demand",
        commodity = "electricity",
        hasCapacityVariable = False,
        operationRateFix = electricityDemand
    )
)

In this example, the electricity demand is synthetically generated for demonstration purposes; in a typical energy system model, this data would instead come from real demand datasets or projections. These inputs must conform to one of the accepted formats as described [below](#operationratefix).

## Save the Energy System Model

In [3]:
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink.nc"

xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=nc_file, overwriteExisting=True
)


Writing output to netCDF... 
Done. (0.2540 sec)


## General Structure of a Sink Instance

The Sink class inherits from the Source class; they share the same input parameters (see [Source class](_1_add_source.ipynb#required-arguments) for the parameter description) and differ only in the sign parameter, which is equal to $-1$ for Sink objects and $+1$ for Source objects.


## List of all parameters

Below, after executing the code cell, you will find the list of all parameters of a Sink, along with their description, type, and default value.

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "NetCDF"))
from docstringTable import display_param_table


display_param_table(fn.Sink)

,Description,Type,Default
Argument,,,
esM,energy system model to which the component should be added. Used for unit checks.,EnergySystemModel instance from the FINE package,/
name,name of the component. Has to be unique (i.e. no other components with that name can already exist in the EnergySystemModel instance to which the component is added).,string,/
commodity,to the component related commodity.,string,/
hasCapacityVariable,specifies if the component should be modeled with a capacity or not. Examples: A wind turbine has a capacity given in GW_electric -> hasCapacityVariable is True. Emitting CO2 into the environment is not per se limited by a capacity -> hasCapacityVariable is False.,boolean,/
capacityVariableDomain,"describes the mathematical domain of the capacity variables, if they are specified. By default, the domain is specified as 'continuous' and thus declares the variables as positive (>=0) real values. The second input option that is available for this parameter is 'discrete', which declares the variables as positive (>=0) integer values.",string ('continuous' or 'discrete'),'continuous'
capacityPerPlantUnit,"capacity of one plant of the component (in the specified physicalUnit of the plant). The default is 1, thus the number of plants is equal to the installed capacity. This parameter should be specified when using a 'discrete' capacityVariableDomain. It can be specified when using a 'continuous' variable domain.",dict of strictly positive float or strictly positive float,1
hasIsBuiltBinaryVariable,"specifies if binary decision variables should be declared for each eligible location of the component, which indicates if the component is built at that location or not (dimension=1dim). each eligible connection of the transmission component, which indicates if the component is built between two locations or not (dimension=2dim). The binary variables can be used to enforce one-time investment cost or capacity-independent annual operation cost. If a minimum capacity is specified and this parameter is set to True, the minimum capacities are only considered if a component is built (i.e. if a component is built at that location, it has to be built with a minimum capacity of XY GW, otherwise it is set to 0 GW).",boolean,False
bigM,"the bigM parameter is only required when the hasIsBuiltBinaryVariable parameter is set to True. In that case, it is set as a strictly positive float, otherwise it can remain a None value. If not None and the ifBuiltBinaryVariables parameter is set to True, the parameter enforces an artificial upper bound on the maximum capacities which should, however, never be reached. The value should be chosen as small as possible but as large as necessary so that the optimal values of the designed capacities are well below this value after the optimization.",None or strictly positive float,None
operationRateMin,"if specified, indicates a minimum operation rate for each location and each time, if required also for each investment period, if step by a positive float. If hasCapacityVariable is set to True, the values are given relative to the installed capacities (i.e. a value of 1 indicates a utilization of 100% of the capacity). If hasCapacityVariable is set to False, the values are given as absolute values in form of the commodityUnit for each time step.",None Pandas DataFrame with positive (>= 0) entries. The row indices have to match the in the energy system model specified time steps. The column indices have to equal the in the energy system model specified locations. The data in ineligible locations are set to zero. a dict,None
